# 🚀 Convert Fine-Tuned IELTS Copilot to 4-bit GGUF

This notebook converts our fine-tuned `TemurbekHamzaev/qwen2.5-1.5b-ielts-copilot` model into a compact **4-bit GGUF (`Q4_K_M`)** file.

### Why GGUF?
- **Size:** Reduced from 3.09 GB (FP16) down to **~1.1 GB**.
- **Memory:** Uses **< 1.5 GB RAM**.
- **Speed:** Runs natively on Apple Silicon (M1/M2/M3 Metal) and standard CPUs at **40+ tokens/sec**.
- **Compatibility:** Runs directly in Ollama, LM Studio, or `llama-cpp-python`.

### Step 1: Install llama.cpp & Dependencies

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!pip install -q huggingface_hub

### Step 2: Build `llama-quantize` Binary (C++)

In [ ]:
!cmake -B llama.cpp/build -S llama.cpp
!cmake --build llama.cpp/build --config Release -j --target llama-quantize

### Step 3: Download Model / Point to Merged Directory & Convert to FP16 GGUF

In [ ]:
import os
from huggingface_hub import snapshot_download

# If running in the same Colab where checkpoints/qwen_ielts_merged exists, use it.
# Otherwise, download directly from your Hugging Face repo.
if os.path.exists("checkpoints/qwen_ielts_merged"):
    model_dir = "checkpoints/qwen_ielts_merged"
    print("Using local merged directory:", model_dir)
else:
    print("Downloading merged model from Hugging Face...")
    model_dir = snapshot_download(repo_id="TemurbekHamzaev/qwen2.5-1.5b-ielts-copilot")
    print("Downloaded to:", model_dir)

# Convert Hugging Face Safetensors to intermediate FP16 GGUF
!python3 llama.cpp/convert_hf_to_gguf.py {model_dir} \
  --outfile qwen_ielts_f16.gguf \
  --outtype f16

### Step 4: Quantize to 4-bit (`Q4_K_M`)

In [ ]:
# Quantize intermediate FP16 GGUF to Q4_K_M (the gold-standard 4-bit format)
!./llama.cpp/build/bin/llama-quantize qwen_ielts_f16.gguf qwen2.5-1.5b-ielts-q4_k_m.gguf q4_k_m

# Check resulting file sizes
!ls -lh qwen_ielts_f16.gguf qwen2.5-1.5b-ielts-q4_k_m.gguf

### Step 5: Upload the 4-bit GGUF to Hugging Face Hub

In [ ]:
import os
from huggingface_hub import HfApi, login

# Enter your HF write token when prompted or set HF_TOKEN environment variable
login(token=os.environ.get("HF_TOKEN"))
HF_REPO_ID = "TemurbekHamzaev/qwen2.5-1.5b-ielts-copilot"
api = HfApi()

print(f"🚀 Uploading 4-bit GGUF (~1.1 GB) to https://huggingface.co/{HF_REPO_ID}...")
api.upload_file(
    path_or_fileobj="qwen2.5-1.5b-ielts-q4_k_m.gguf",
    path_in_repo="qwen2.5-1.5b-ielts-q4_k_m.gguf",
    repo_id=HF_REPO_ID,
    repo_type="model",
)

print("\n" + "=" * 65)
print(f"🎉 SUCCESS! 4-bit GGUF model is live at:")
print(f"👉 https://huggingface.co/{HF_REPO_ID}/blob/main/qwen2.5-1.5b-ielts-q4_k_m.gguf")
print("=" * 65)